# Across Sessions + Learning Curves Example

This notebook is a minimal front-end over the existing analysis code. Select a line/cohort/genotype/animal at the top, then run the plotting cells.

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks.example_analysis_helpers import (
    DatasetSelection,
    available_animals,
    available_genotypes,
    build_selection_table,
    discover_datasets,
    filter_subjects,
    load_dataset,
    plot_across_sessions,
    plot_learning_curves,
    prepare_across_sessions_data,
    compute_learning_curve_view,
)


In [ ]:
datasets = discover_datasets()
datasets[['line', 'cohort']]

In [ ]:
line_widget = widgets.Dropdown(options=sorted(datasets['line'].unique()), description='Line')
cohort_widget = widgets.Dropdown(description='Cohort')
genotype_widget = widgets.Dropdown(description='Genotype')
animal_widget = widgets.Dropdown(description='Animal')

def _refresh_cohorts(*_):
    cohorts = sorted(datasets.loc[datasets['line'] == line_widget.value, 'cohort'].unique())
    cohort_widget.options = cohorts
    cohort_widget.value = cohorts[0] if cohorts else None

def _refresh_subject_options(*_):
    selection = DatasetSelection(line=line_widget.value, cohort=cohort_widget.value)
    _, meta, _ = load_dataset(selection)
    genotype_widget.options = available_genotypes(meta)
    if genotype_widget.value not in genotype_widget.options:
        genotype_widget.value = genotype_widget.options[0]
    animal_widget.options = available_animals(meta, genotype_widget.value)
    if animal_widget.value not in animal_widget.options:
        animal_widget.value = animal_widget.options[0]

def _refresh_animals(*_):
    selection = DatasetSelection(line=line_widget.value, cohort=cohort_widget.value)
    _, meta, _ = load_dataset(selection)
    animal_widget.options = available_animals(meta, genotype_widget.value)
    if animal_widget.value not in animal_widget.options:
        animal_widget.value = animal_widget.options[0]

line_widget.observe(_refresh_cohorts, names='value')
cohort_widget.observe(_refresh_subject_options, names='value')
genotype_widget.observe(_refresh_animals, names='value')

_refresh_cohorts()
_refresh_subject_options()

display(widgets.HBox([line_widget, cohort_widget, genotype_widget, animal_widget]))

In [ ]:
selection = DatasetSelection(
    line=line_widget.value,
    cohort=cohort_widget.value,
    genotype=genotype_widget.value,
    animal=animal_widget.value,
)

df_raw, meta, data_dir = load_dataset(selection)
meta_table = build_selection_table(meta, df_raw)
df_selected = filter_subjects(df_raw, meta_table, selection)

print(f'Data dir: {data_dir}')
print(f'Selected rows: {len(df_selected):,}')
print(f'Animals: {sorted(df_selected["animal"].astype(str).unique())}')
meta_table.head()

## Across Sessions

In [ ]:
across_summaries = prepare_across_sessions_data(df_selected)
change_points_csv = data_dir / 'change_points.csv'
subject_for_shading = selection.animal if selection.animal != 'all' else None

fig = plot_across_sessions(
    across_summaries,
    title=f'Across sessions: {selection.line} {selection.cohort} | genotype={selection.genotype} | animal={selection.animal}',
    change_points_csv=change_points_csv,
    subject=subject_for_shading,
)


## Learning Curves

In [ ]:
learning_results = compute_learning_curve_view(df_selected, meta_table, selection)
fig = plot_learning_curves(
    learning_results,
    title=f'Learning curves: {selection.line} {selection.cohort} | genotype={selection.genotype} | animal={selection.animal}',
)


## Notes for extending this notebook

- Reuse the same `selection` object for `DailyPlots`, `groupComparison`, `stimDur`, `GLM`, and the histogram analyses.
- Keep the analysis logic in helper modules and let the notebook only handle parameters and plotting.
- If you want one-button execution later, these functions can also sit behind a small GUI or Streamlit app.